In [30]:
import pandas as pd

df_movies = pd.read_csv('/content/netflix_cleaned.csv')

print("5 baris pertama dataset:")
display(df_movies.head())

5 baris pertama dataset:


,Title,Genre,parent_genre,parent_genre_str,metadata,poster,IMDB Score,Runtime,Language,Premiere
0,enter the anime,documentary,['documentary'],documentary,movie title is enter the anime. genre is docum...,https://m.media-amazon.com/images/M/MV5BNzljM2...,2.5,58,english/japanese,"August 5, 2019"
1,dark forces,thriller,['action'],action,movie title is dark forces. genre is action. l...,https://m.media-amazon.com/images/M/MV5BOTIxMW...,2.6,81,spanish,"August 21, 2020"
2,the app,science fiction drama,"['scifi', 'drama']","scifi, drama","movie title is the app. genre is scifi, drama....",https://m.media-amazon.com/images/M/MV5BNzgzZG...,2.6,79,italian,"December 26, 2019"
3,the open house,horror thriller,"['action', 'horror']","action, horror",movie title is the open house. genre is action...,https://m.media-amazon.com/images/M/MV5BMTU0Mj...,3.2,94,english,"January 19, 2018"
4,kaali khuhi,mystery,['mystery'],mystery,movie title is kaali khuhi. genre is mystery. ...,https://m.media-amazon.com/images/M/MV5BZjRhNG...,3.4,90,hindi,"October 30, 2020"


In [31]:
num_unique_genres = df_movies['parent_genre'].nunique()
print(f"Jumlah kategori genre unik di dataset adalah: {num_unique_genres}")

Jumlah kategori genre unik di dataset adalah: 54


In [32]:
unique_genres = df_movies['parent_genre'].unique()
print("Daftar kategori genre unik:")
for genre in unique_genres:
    print(f"- {genre}")

Daftar kategori genre unik:
- ['documentary']
- ['action']
- ['scifi', 'drama']
- ['action', 'horror']
- ['mystery']
- ['comedy']
- ['fantasy', 'western', 'musical']
- ['drama']
- ['comedy', 'romance']
- ['comedy', 'action']
- ['horror']
- ['romance', 'drama']
- ['animation']
- ['western']
- ['animation', 'action']
- ['family']
- ['comedy', 'drama']
- ['adventure', 'scifi']
- ['scifi']
- ['other']
- ['comedy', 'fantasy', 'family']
- ['scifi', 'action']
- ['comedy', 'musical']
- ['musical']
- ['scifi', 'mystery']
- ['drama', 'action']
- ['comedy', 'adventure']
- ['romance']
- ['comedy', 'horror']
- ['comedy', 'romance', 'drama']
- ['fantasy']
- ['comedy', 'mystery']
- ['romance', 'action']
- ['comedy', 'family']
- ['comedy', 'war']
- ['comedy', 'romance', 'family']
- ['adventure', 'romance']
- ['adventure']
- ['drama', 'action', 'horror']
- ['drama', 'horror']
- ['comedy', 'drama', 'family']
- ['war']
- ['musical', 'documentary']
- ['adventure', 'animation', 'musical']
- ['comedy', 'adv

In [33]:
print("\nInformasi dataset:")
display(df_movies.info())


Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Title             528 non-null    object 
 1   Genre             528 non-null    object 
 2   parent_genre      528 non-null    object 
 3   parent_genre_str  528 non-null    object 
 4   metadata          528 non-null    object 
 5   poster            528 non-null    object 
 6   IMDB Score        528 non-null    float64
 7   Runtime           528 non-null    int64  
 8   Language          528 non-null    object 
 9   Premiere          528 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 41.4+ KB


None

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# Fit dan transform kolom 'parent_genre_str' untuk membuat matriks TF-IDF
tfidf_matrix = tfidf_vectorizer.fit_transform(df_movies['parent_genre_str'])

print(f"Bentuk matriks TF-IDF untuk genre: {tfidf_matrix.shape}")

Bentuk matriks TF-IDF untuk genre: (528, 15)


In [35]:
print("Nilai-nilai pertama dari matriks TF-IDF (dikonversi ke dense array untuk tampilan):")
# Convert to a dense array to print, but only show a small slice as it can be very large
display(tfidf_matrix.toarray()[:5, :5])

Nilai-nilai pertama dari matriks TF-IDF (dikonversi ke dense array untuk tampilan):


array([[0.        , 0.        , 0.        , 0.        , 1.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.56219348, 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ]])

In [36]:
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

# Inisialisasi MinMaxScaler
scaler = MinMaxScaler()

# Melakukan scaling pada fitur numerik: 'IMDB Score' dan 'Runtime'
# reshape(-1, 1) diperlukan karena scaler mengharapkan input 2D
numeric_features = scaler.fit_transform(df_movies[['IMDB Score', 'Runtime']])

print(f"Bentuk matriks fitur numerik setelah scaling: {numeric_features.shape}")

# Gabungkan matriks TF-IDF dengan fitur numerik yang sudah discale
feature_matrix = hstack([tfidf_matrix, numeric_features])

print(f"Bentuk matriks fitur akhir (genre TF-IDF + numerik): {feature_matrix.shape}")

Bentuk matriks fitur numerik setelah scaling: (528, 2)
Bentuk matriks fitur akhir (genre TF-IDF + numerik): (528, 17)


In [37]:
from sklearn.metrics.pairwise import cosine_similarity

# Menghitung cosine similarity dari matriks fitur akhir
cosine_sim = cosine_similarity(feature_matrix)

print(f"Bentuk matriks cosine similarity: {cosine_sim.shape}")

print("\nBeberapa nilai pertama dari matriks cosine similarity:")
print(cosine_sim[:5, :5])

Bentuk matriks cosine similarity: (528, 528)

Beberapa nilai pertama dari matriks cosine similarity:
[[1.         0.08955826 0.08750984 0.10190288 0.09774703]
 [0.08955826 1.         0.12099436 0.62158664 0.1367402 ]
 [0.08750984 0.12099436 1.         0.13885305 0.13366013]
 [0.10190288 0.62158664 0.13885305 1.         0.16594174]
 [0.09774703 0.1367402  0.13366013 0.16594174 1.        ]]


In [38]:
import numpy as np
# Buat Series dari judul film dan indeksnya untuk memudahkan pencarian
indices = pd.Series(df_movies.index, index=df_movies['Title']).drop_duplicates()

def get_recommendations(title, cosine_sim_matrix, df, num_recommendations=10):
    """
    Fungsi untuk mendapatkan rekomendasi film berdasarkan judul film.

    Args:
        title (str): Judul film yang ingin dicari rekomendasinya.
        cosine_sim_matrix (np.array): Matriks cosine similarity.
        df (pd.DataFrame): DataFrame yang berisi informasi film.
        num_recommendations (int): Jumlah rekomendasi film yang diinginkan.

    Returns:
        pd.DataFrame: DataFrame berisi film-film yang direkomendasikan.
    """
    # Dapatkan indeks film yang cocok dengan judul
    if title not in indices:
        print(f"Film '{title}' tidak ditemukan dalam dataset. Mohon periksa kembali judulnya.")
        return pd.DataFrame()

    idx = indices[title]

    # Dapatkan skor similarity dari film tersebut dengan semua film lain
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Urutkan film berdasarkan skor similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Ambil skor dari 10 film teratas (kecuali film itu sendiri)
    # Pastikan untuk menghindari film itu sendiri (indeks 0 setelah diurutkan)
    sim_scores = sim_scores[1:num_recommendations+1]

    # Dapatkan indeks film-film yang direkomendasikan
    movie_indices = [i[0] for i in sim_scores]

    # Kembalikan judul, genre, IMDB Score, Runtime, poster, Language, dan Premiere dari film-film yang direkomendasikan
    return df[['Title', 'parent_genre_str', 'IMDB Score', 'Runtime', 'poster', 'Language', 'Premiere']].iloc[movie_indices]


In [39]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widget untuk input judul film
film_input = widgets.Text(
    value='the app', # Default value
    placeholder='Masukkan judul film...',
    description='Judul Film:',
    disabled=False
)

# Widget untuk tombol rekomendasi
recommend_button = widgets.Button(
    description='Dapatkan Rekomendasi',
    disabled=False,
    button_style='info',
    tooltip='Klik untuk mendapatkan rekomendasi film',
    icon='search'
)

# Widget untuk menampilkan output
output_area = widgets.Output()

# Fungsi yang akan dijalankan saat tombol diklik
def on_recommend_button_clicked(b):
    with output_area:
        clear_output()
        title = film_input.value.strip().lower() # Pastikan case-insensitive dan tanpa spasi ekstra

        # Cek apakah judul film ada di dataset
        if title not in df_movies['Title'].str.lower().values:
            print(f"Film '{title}' tidak ditemukan dalam dataset. Mohon periksa kembali judulnya atau gunakan judul lengkap yang tersedia.")
            return

        # Ambil judul asli dengan case yang benar dari dataset untuk pencarian indeks
        original_title = df_movies[df_movies['Title'].str.lower() == title]['Title'].iloc[0]

        print(f"\nMemproses rekomendasi untuk film: '{original_title}'")

        # Tambahkan informasi genre asli film input
        if original_title in indices:
            print(f"  Index film '{original_title}': {indices[original_title]}")
            print(f"  Genre asli '{original_title}': {df_movies.loc[indices[original_title], 'parent_genre_str']}")

        recommendations = get_recommendations(original_title, cosine_sim, df_movies, num_recommendations=10)
        if not recommendations.empty:
            print("\nBerikut adalah rekomendasi film untuk Anda:")
            display(recommendations)

# Hubungkan tombol dengan fungsi
recommend_button.on_click(on_recommend_button_clicked)

# Tampilkan widget
display(film_input, recommend_button, output_area)

Text(value='the app', description='Judul Film:', placeholder='Masukkan judul film...')

Button(button_style='info', description='Dapatkan Rekomendasi', icon='search', style=ButtonStyle(), tooltip='K…

Output()

### Menyimpan Data untuk API



In [41]:
import joblib


# Simpan tfidf_vectorizer
joblib.dump(tfidf_vectorizer, 'api_tfidf_vectorizer.pkl')
print('api_tfidf_vectorizer.pkl berhasil disimpan.')

# Simpan feature_matrix (sparse matrix)
joblib.dump(feature_matrix, 'api_feature_matrix.pkl')
print('api_feature_matrix.pkl berhasil disimpan.')

# Simpan DataFrame film (hanya kolom yang relevan untuk rekomendasi)
df_movies[['Title', 'parent_genre_str', 'IMDB Score', 'Runtime', 'poster', 'Language', 'Premiere']].to_pickle('api_df_movies.pkl')
print('api_df_movies.pkl berhasil disimpan.')

api_tfidf_vectorizer.pkl berhasil disimpan.
api_feature_matrix.pkl berhasil disimpan.
api_df_movies.pkl berhasil disimpan.
